# 01 - HYCOM Water Temperature Anomalies

Computes daily water temperature anomalies from HYCOM data (2015-2024).

**Steps:**
1. Load daily HYCOM surface temperature files for each year
2. Compute a day-of-year climatology (mean across all years)
3. Subtract the climatology to obtain daily anomalies
4. Save the climatology and per-year anomaly files to the features directory

In [ ]:
import xarray as xr
from pathlib import Path
from pyproj import datadir
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")

data_path = Path("/home/jupyter-daniela/suyana/peru_production/features/")
years = range(2015, 2025)

In [3]:
datasets = []
for year in years:
    fpath = data_path / f"hycom_water_temp_daily_{year}.nc"
    ds = xr.open_dataset(fpath)
    datasets.append(ds)
    print(f"{year}: {ds.sizes['time']} days loaded")

ds_all = xr.concat(datasets, dim='time')
ds_all = ds_all.sortby('time')

ds_all = ds_all.squeeze('depth', drop=True)

print(f"\nTotal: {ds_all.sizes['time']} days | lat: {ds_all.sizes['lat']} | lon: {ds_all.sizes['lon']}")
ds_all

2015: 365 days loaded
2016: 366 days loaded
2017: 365 days loaded
2018: 365 days loaded
2019: 365 days loaded
2020: 366 days loaded
2021: 365 days loaded
2022: 365 days loaded
2023: 365 days loaded
2024: 249 days loaded


/tmp/ipykernel_1086315/969437631.py:8: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'lat' ('lat',) The recommendation is to set join explicitly for this case.
  ds_all = xr.concat(datasets, dim='time')
/tmp/ipykernel_1086315/969437631.py:8: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'lon' ('lon',) The recommendation is to set join explicitly for this case.
  ds_all = xr.concat(datasets, dim='time')



Total: 3536 days | lat: 501 | lon: 502


<xarray.Dataset> Size: 4GB
Dimensions:     (time: 3536, lat: 501, lon: 502)
Coordinates:
  * lat         (lat) float64 4kB -20.0 -19.96 -19.92 -19.88 ... -0.08 -0.04 0.0
  * lon         (lon) float64 4kB -90.0 -89.92 -89.84 ... 289.8 289.9 290.0
  * time        (time) datetime64[ns] 28kB 2015-01-01 2015-01-02 ... 2024-09-05
Data variables:
    water_temp  (time, lat, lon) float32 4GB 20.84 20.82 20.79 ... nan nan nan
Attributes: (12/14)
    classification_level:      UNCLASSIFIED
    distribution_statement:    Approved for public release. Distribution unli...
    downgrade_date:            not applicable
    classification_authority:  not applicable
    institution:               Naval Oceanographic Office
    source:                    HYCOM archive file
    ...                        ...
    Conventions:               CF-1.6 NAVO_netcdf_v1.1
    History:                   Translated to CF-1.0 Conventions by Netcdf-Jav...
    geospatial_lat_min:        -20.0
    geospatial_lat_max:        0.0
    geospatial_lon_min:        -90.0
    geospatial_lon_max:        -70.0

In [4]:
climatology = ds_all['water_temp'].groupby('time.dayofyear').mean('time')
print("Climatology shape:", climatology.shape)
print("Day-of-year range:", int(climatology.dayofyear.min()), "–", int(climatology.dayofyear.max()))


Climatology shape: (366, 501, 502)
Day-of-year range: 1 – 366


In [5]:
anomalies = ds_all['water_temp'].groupby('time.dayofyear') - climatology
anomalies.name = 'water_temp_anomaly'
print("Anomalies shape:", anomalies.shape)


Anomalies shape: (3536, 501, 502)


In [6]:
out_path = Path("/home/jupyter-daniela/suyana/peru_production/features/")

climatology.to_dataset(name='water_temp_climatology').to_netcdf(out_path / "hycom_water_temp_climatology.nc")
print("Saved: hycom_water_temp_climatology.nc")

for year in years:
    anom_year = anomalies.sel(time=anomalies.time.dt.year == year)
    anom_year.to_dataset(name='water_temp_anomaly').to_netcdf(out_path / f"hycom_water_temp_anomaly_daily_{year}.nc")
    print(f"Saved: hycom_water_temp_anomaly_daily_{year}.nc  ({anom_year.sizes['time']} days)")


Saved: hycom_water_temp_climatology.nc
Saved: hycom_water_temp_anomaly_daily_2015.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2016.nc  (366 days)
Saved: hycom_water_temp_anomaly_daily_2017.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2018.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2019.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2020.nc  (366 days)
Saved: hycom_water_temp_anomaly_daily_2021.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2022.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2023.nc  (365 days)
Saved: hycom_water_temp_anomaly_daily_2024.nc  (249 days)
